# 확률분포와 손실함수: MLE에서 유도하기

[Step 6 이슈 #9](https://github.com/ntt65/A2_1/issues/9). 수학 계산은 NumPy, 그림은 Matplotlib로 수행한다.
정규분포 표기는 **$N(\mu,\sigma^2)$: 평균과 분산**이다. 모든 로그는 자연로그이며 손실의 **합과 평균**을 구분한다.

정규·베르누이 분포의 질량을 수치로 확인한 뒤 Gaussian, Bernoulli, categorical likelihood에서 각 손실을 유도한다.
새 커널에서 위에서 아래로 실행한다. 독립 고정값·해석식과 명시적 허용오차를 assertion으로 검증하며, 확률을 임의로 clipping하지 않는다.

In [1]:
# 파일 개요: 분포·확률 정규화·우도와 손실의 관계를 직접 계산하고 검증 결과와 그림을 저장한다.
import json
import os
from pathlib import Path
import sys

import numpy as np
import matplotlib
matplotlib.use("Agg")	# 평가 화면이 없는 환경에서도 같은 그림을 파일로 만들 수 있게 한다.
import matplotlib.pyplot as plt

repo_root = Path.cwd().resolve()	# 실행 위치를 절대 경로로 고정해 저장소 기준 경로를 찾는다.
if not (repo_root / "src" / "utils" / "math.py").is_file():
    candidate = repo_root.parent	# notebooks 폴더에서 실행했다면 한 단계 위를 후보로 삼는다.
    if repo_root.name == "notebooks" and (candidate / "src" / "utils" / "math.py").is_file():
        repo_root = candidate	# src가 실제로 있는 상위 폴더를 저장소 루트로 선택한다.
    else:
        raise RuntimeError("Run from the A2_1 root or its notebooks directory")
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))	# 현재 저장소의 공통 함수를 가져올 수 있도록 경로를 한 번 추가한다.

from src.utils.math import sigmoid, softmax, _as_finite_real_array
from src.utils.numerical import _finite_scalar
from src.utils.random import set_seed
from src.calculus import numerical_gradient

set_seed(42)	# 독립 실험의 시작점에서만 공통 난수 시드를 한 번 설정한다.
output_dir = Path(os.environ.get("A2_1_OUTPUT_DIR", str(repo_root / "outputs"))).resolve()	# 검증에서는 임시 경로를 지정할 수 있고 기본 결과는 outputs에 저장한다.
output_dir.mkdir(parents=True, exist_ok=True)	# 출력 디렉터리를 준비하되 기존 파일을 삭제하지 않는다.
print(f"NumPy {np.__version__}; float64; seed=42; deterministic grids")

NumPy 2.2.6; float64; seed=42; deterministic grids


## 1. 정규분포 PDF

$$p(x\mid\mu,\sigma^2)=\frac{1}{\sqrt{2\pi}\sigma}
\exp\!\left(-\frac{(x-\mu)^2}{2\sigma^2}\right),\qquad \sigma^2>0.$$

`N(0,1)`의 표준편차는1, `N(2,0.5)`의 표준편차는 $\sqrt{0.5}$다. 분산0.5를 표준편차0.5로 해석하지 않는다.
PDF는 확률 **밀도**이므로 개별 높이가1보다 클 수 있으며, 전체 적분이1이다.
각 분포의 $\mu\pm8\sigma$ 구간을10,001점으로 나누어 사다리꼴 적분한다. 유한 구간의 꼬리 누락과 수치 적분 오차가 있으므로 정확히1이라고 강제하지 않고 절대오차 $\le10^{-4}$를 검사한다.

In [2]:
def normal_pdf(x, mu, variance):
    """스칼라 평균과 양의 분산으로 x와 같은 shape의 정규분포 밀도를 반환한다.

    입력은 유한 실수 float64로 처리하며 mu와 variance의 shape는 ()다.
    실수가 아닌 입력은 TypeError, 잘못된 스칼라 매개변수나 분산은 ValueError다.
    극단적인 유한 꼬리에서는 float64 표현 한계 때문에 밀도가 0으로 언더플로될
    수 있다. 입력을 변경하지 않으며 표현할 수 없는 비유한 밀도는 거부한다.
    """
    values = _as_finite_real_array(x, "x")	# 밀도를 계산할 입력의 실수·유한성 계약을 공통 함수로 검사한다.
    location = _finite_scalar(mu, "mu")	# 평균은 shape ()의 유한한 실수여야 한다.
    spread = _finite_scalar(variance, "variance")	# N(mu, variance)의 두 번째 분포 매개변수는 표준편차가 아니라 분산이다.
    if spread <= 0:	# 정규분포의 분산은 엄격히 양수여야 한다.
        raise ValueError("variance must be positive")
    sigma = np.sqrt(spread)	# 밀도식에 사용할 표준편차는 분산의 제곱근이다.
    with np.errstate(over="ignore", under="ignore"):
        standardized = (values-location)/sigma	# 평균을 빼고 표준편차로 나누어 표준화한다.
        density = np.exp(-0.5*standardized**2)/(np.sqrt(2*np.pi)*sigma)	# 정규화 계수를 포함한 정규분포 PDF를 계산한다.
    if not np.all(np.isfinite(density)):	# 허용한 꼬리의 0과 달리 비유한 밀도는 표현 실패로 거부한다.
        raise ValueError("density must be representable as finite float64")
    return np.asarray(density)	# 스칼라를 포함해 입력과 같은 shape의 밀도를 반환한다.


normals = []
for mu, variance, expected_peak in ((0.0, 1.0, 0.3989422804014327),
                                   (2.0, 0.5, 0.5641895835477563)):
    sigma = np.sqrt(variance)	# 실험 격자의 폭도 분산이 아닌 표준편차를 기준으로 정한다.
    grid = np.linspace(mu-8*sigma, mu+8*sigma, 10001)	# 평균 양옆 8표준편차의 고정 격자에서 밀도 적분을 근사한다.
    density = normal_pdf(grid, mu, variance)	# 같은 검증 함수를 사용해 격자의 밀도값을 구한다.
    integral = float(np.trapezoid(density, grid))	# PDF 높이의 합이 아닌 곡선 아래 면적을 수치 적분한다.
    peak = float(normal_pdf(mu, mu, variance))	# 평균 위치의 봉우리 높이를 독립 기준과 비교한다.
    assert density.shape == (10001,) and np.all(density >= 0)	# 격자 크기와 밀도 비음수 조건을 확인한다.
    assert abs(integral-1.0) <= 1e-4	# 유한 구간의 수치 적분이 확률 1에 충분히 가까운지 확인한다.
    assert abs(peak-expected_peak) <= 1e-15	# 닫힌 식에서 얻은 독립 봉우리 상수와 비교한다.
    np.testing.assert_allclose(normal_pdf([mu-sigma, mu+sigma], mu, variance),
                               np.full(2, expected_peak*np.exp(-0.5)), atol=1e-15, rtol=0)
    normals.append({"mu": mu, "variance": variance, "sigma": float(sigma),
                    "grid_points": len(grid), "integral": integral,
                    "integral_absolute_error": abs(integral-1.0), "peak": peak,
                    "minimum_pdf": float(np.min(density))})
for invalid in (0.0, -1.0, np.inf, np.nan):
    try:
        normal_pdf([0.0], 0.0, invalid)
    except ValueError:
        pass
    else:
        raise AssertionError("invalid variance was accepted")
for item in normals:
    print(item)

figure, axes = plt.subplots(figsize=(8.2, 4.8), layout="constrained")
try:
    display_x = np.linspace(-5.0, 6.0, 1001)
    for item in normals:
        axes.plot(display_x, normal_pdf(display_x, item["mu"], item["variance"]),
                  label=f"N(mu={item['mu']:g}, variance={item['variance']:g})")
    axes.set(xlabel="x", ylabel="Probability density", title="Normal densities: second parameter is variance")
    axes.legend()	# 비교 조건을 구분할 수 있도록 범례를 표시한다.
    axes.grid(alpha=0.25)
    figure.savefig(output_dir / "probability_normal.png", dpi=140)	# 축과 범례를 포함한 기존 정규분포 그림 경로를 유지한다.
finally:
    plt.close(figure)	# 성공 여부와 관계없이 이번에 만든 Figure 자원을 해제한다.


{'mu': 0.0, 'variance': 1.0, 'sigma': 1.0, 'grid_points': 10001, 'integral': 0.9999999999999989, 'integral_absolute_error': 1.1102230246251565e-15, 'peak': 0.3989422804014327, 'minimum_pdf': 5.052271083536893e-15}
{'mu': 2.0, 'variance': 0.5, 'sigma': 0.7071067811865476, 'grid_points': 10001, 'integral': 0.999999999999999, 'integral_absolute_error': 9.992007221626409e-16, 'peak': 0.5641895835477563, 'minimum_pdf': 7.144990287123286e-15}


![평균과 분산이 표시된 정규분포](../outputs/probability_normal.png)

폭과 높이의 차이를 확인한다. 높은 PDF 값 자체는 개별 점의 확률이 아니다.

## 2. 베르누이 PMF

$$P(Y=y)=p^y(1-p)^{1-y},\qquad y\in\{0,1\},\quad 0\le p\le1.$$

support는0과1이며 확률벡터는 `[1-p,p]`다. `p=.3`은 `[.7,.3]`, `p=.7`은 `[.3,.7]`이고 합은1이다.
연속 밀도와 달리 PMF는 각 support의 확률 자체다.

In [3]:
def bernoulli_pmf(p):
    """[0,1]의 스칼라 p에서 지지집합 [0,1]의 확률을 shape (2,)로 반환한다.

    유한한 실수 스칼라를 받아 float64 배열 [1-p,p]를 만든다.
    잘못된 자료형은 TypeError이며 잘못된 shape, 비유한 p 또는 [0,1] 밖의
    확률은 ValueError로 거부한다.
    """
    probability = _finite_scalar(p, "p")	# 성공 확률은 유한한 실수 스칼라로 제한한다.
    if not 0 <= probability <= 1:	# 경계 0과 1은 허용하고 그 밖의 확률은 거부한다.
        raise ValueError("p must be in [0,1]")
    return np.array([1-probability, probability], dtype=np.float64)	# 지지집합 0과 1의 순서에 맞춰 실패·성공 확률을 반환한다.


bernoulli = []
for p, reference in ((0.3, [0.7, 0.3]), (0.7, [0.3, 0.7])):
    pmf = bernoulli_pmf(p)	# 각 p에 대한 두 확률 질량을 계산한다.
    np.testing.assert_allclose(pmf, reference, atol=1e-15, rtol=0)	# 미리 정한 독립 확률 벡터와 원소별로 비교한다.
    assert np.all((pmf >= 0) & (pmf <= 1))	# 확률 질량의 범위를 확인한다.
    mass_error = abs(float(np.sum(pmf))-1.0)	# 이산 분포는 지지집합의 확률 합으로 정규화를 검사한다.
    assert mass_error <= 1e-12	# 기존 확률 합 허용오차를 그대로 적용한다.
    bernoulli.append({"p": p, "support": [0,1], "pmf": pmf.tolist(), "mass_error": mass_error})
np.testing.assert_array_equal(bernoulli_pmf(0), [1,0])	# 항상 실패하는 경계 분포를 정확히 확인한다.
np.testing.assert_array_equal(bernoulli_pmf(1), [0,1])	# 항상 성공하는 경계 분포를 정확히 확인한다.
for invalid in (-0.1, 1.1, np.nan):
    try:
        bernoulli_pmf(invalid)
    except ValueError:
        pass
    else:
        raise AssertionError("invalid Bernoulli probability was accepted")
print(bernoulli)

figure, axes = plt.subplots(figsize=(7.2, 4.6), layout="constrained")
try:
    support = np.array([0,1])
    for shift, item in zip((-0.18, 0.18), bernoulli):
        bars = axes.bar(support+shift, item["pmf"], width=0.34, label=f"p={item['p']:g}")
        axes.bar_label(bars, fmt="%.1f", padding=3)
    axes.set(xticks=support, xlabel="Outcome y", ylabel="Probability mass",
             ylim=(0,1), title="Bernoulli probability mass on {0,1}")
    axes.legend()	# 비교 조건을 구분할 수 있도록 범례를 표시한다.
    axes.grid(axis="y", alpha=0.2)
    figure.savefig(output_dir / "probability_bernoulli.png", dpi=140)	# 기존 Bernoulli 그림 링크가 가리키는 파일명을 유지한다.
finally:
    plt.close(figure)	# 성공 여부와 관계없이 이번에 만든 Figure 자원을 해제한다.


[{'p': 0.3, 'support': [0, 1], 'pmf': [0.7, 0.3], 'mass_error': 0.0}, {'p': 0.7, 'support': [0, 1], 'pmf': [0.30000000000000004, 0.7], 'mass_error': 0.0}]


![베르누이 PMF 비교](../outputs/probability_bernoulli.png)

## 3. Softmax: 같은 상수를 빼도 확률은 같다

$$p_j=\frac{e^{z_j}}{\sum_k e^{z_k}}
=\frac{e^{z_j-m}}{\sum_k e^{z_k-m}},\qquad m=\max_k z_k.$$

공통 모듈을 재사용한다. 아래 `(2,3)` 배열은 행마다 하나의 범주 분포이며 `axis=-1`로 정규화한다.
다른 축을 지정하면 정규화되는 분포 묶음도 바뀐다. 유한한 큰 logit에서도 확률합 오차 $\le10^{-6}$인지 확인한다.

In [4]:
logits = np.array([[1000.0, 1001.0, 1002.0], [-1000.0, 0.0, 1000.0]])
probabilities = softmax(logits, axis=-1)	# 각 행의 마지막 클래스 축을 따라 확률을 정규화한다.
row_sums = np.sum(probabilities, axis=-1)	# 정규화한 것과 같은 축에서 합을 확인한다.
assert probabilities.shape == logits.shape	# Softmax가 입력의 축과 원소 배치를 유지하는지 확인한다.
assert np.all(np.isfinite(probabilities)) and np.all(probabilities >= 0)	# 큰 logit에도 overflow나 음수 확률이 생기지 않아야 한다.
max_sum_error = float(np.max(np.abs(row_sums-1)))	# 행마다 다른 정규화 오차 중 최댓값을 기록한다.
assert max_sum_error <= 1e-6	# 미션의 확률 합 허용치와 직접 비교한다.
np.testing.assert_allclose(probabilities[0], [0.09003057317038046, 0.24472847105479764, 0.6652409557748218], atol=1e-15, rtol=0)
np.testing.assert_allclose(softmax(np.log([2.0,3.0,5.0])), [0.2,0.3,0.5], atol=1e-15, rtol=0)
column_probabilities = softmax(logits, axis=0)	# 열 방향 정규화도 별도로 실행해 축 동작을 검증한다.
column_sum_error = float(np.max(np.abs(column_probabilities.sum(axis=0)-1)))	# 열 정규화 결과는 열 축에서 합을 확인한다.
assert column_sum_error <= 1e-6	# 다른 축에서도 같은 확률 합 기준을 적용한다.
softmax_report = {"logits": logits.tolist(), "probabilities": probabilities.tolist(),
                  "axis": -1, "row_sums": row_sums.tolist(),
                  "max_sum_error": max_sum_error, "column_sum_error": column_sum_error}
print(softmax_report)

{'logits': [[1000.0, 1001.0, 1002.0], [-1000.0, 0.0, 1000.0]], 'probabilities': [[0.09003057317038046, 0.24472847105479764, 0.6652409557748218], [0.0, 0.0, 1.0]], 'axis': -1, 'row_sums': [0.9999999999999999, 1.0], 'max_sum_error': 1.1102230246251565e-16, 'column_sum_error': 1.1102230246251565e-16}


## 4. Gaussian MLE가 MSE 최소화와 연결되는 조건

관측 $y_i=f_\theta(x_i)+\epsilon_i$에서 오차는 서로 독립이며 $\epsilon_i\sim N(0,\sigma^2)$라고 가정한다.
모든 표본에 같은 양의 분산 $\sigma^2$를 사용하며, 여기서는 그 분산을 **알려진 상수**로 고정한다.

$$\mathcal L(\theta)=\prod_{i=1}^n\frac{1}{\sqrt{2\pi\sigma^2}}
\exp\!\left[-\frac{(y_i-f_\theta(x_i))^2}{2\sigma^2}\right].$$

곱의 자연로그를 합으로 바꾸고 부호를 반전하면

$$\operatorname{NLL}(\theta)
=\frac n2\ln(2\pi\sigma^2)+\frac{1}{2\sigma^2}\sum_{i=1}^n(y_i-f_\theta(x_i))^2
=\frac n2\ln(2\pi\sigma^2)+\frac{n}{2\sigma^2}\operatorname{MSE}(\theta).$$

첫 항은 $\theta$와 무관하고 MSE의 계수는 양수이므로 같은 허용 매개변수 집합에서 MLE, SSE 최소화, MSE 최소화가 **같은 최적해 집합**을 갖는다.
NLL과 MSE가 항상 같은 숫자라는 뜻은 아니다. 평균 NLL은 위 식을 n으로 나눈 값이며, 양의 상수 배율은 최적해를 바꾸지 않는다.

분산이 입력 또는 $\theta$에 따라 달라지면 로그 분산 항과 오차별 가중치를 버릴 수 없다. 분산까지 추정하는 문제는 위 고정분산 유도와 구분해야 한다.
연속 확률의 밀도는1보다 클 수 있으므로 Gaussian NLL이 음수일 수도 있다. 일반적인 비음수 assertion을 두지 않는다.

In [5]:
observed = np.array([0.0, 1.0, 2.0])
predicted = np.array([1.0, 0.5, 1.5])
variance = 0.5	# Gaussian MLE 비교 동안 양의 공통 분산을 고정한다.
n = observed.size	# 합 손실과 평균 손실을 구분할 표본 수다.
sse = float(np.sum((observed-predicted)**2))	# 잔차 제곱을 모든 표본에 대해 합산한다.
mse = sse/n	# 평균 제곱오차는 SSE를 고정 표본 수로 나눈 값이다.
gaussian_nll = n/2*np.log(2*np.pi*variance)+sse/(2*variance)	# Gaussian 밀도의 정규화 상수와 잔차 항을 함께 유지한다.
independent_gaussian_reference = 3.21709482877410026	# 계산식을 다시 호출하지 않고 독립 계산한 기준값을 사용한다.
assert sse == 1.5 and mse == 0.5	# 고정 입력에서 손으로 확인한 SSE와 MSE를 검증한다.
assert abs(gaussian_nll-independent_gaussian_reference) <= 1e-14	# 유도식의 NLL을 독립 상수와 비교한다.
direct_nll = -float(np.sum([np.log(normal_pdf(y, mean, variance)) for y, mean in zip(observed, predicted)]))	# 밀도를 직접 구해 음의 로그를 더하는 별도 경로로 검산한다.
identity_error = abs(gaussian_nll-direct_nll)	# 유도식과 밀도 계산의 차이를 항등식 오차로 기록한다.
assert identity_error <= 1e-14	# 두 계산 경로의 차이가 기존 허용치 이내인지 확인한다.
assert gaussian_nll != mse	# 최적해가 같다는 주장과 수치 자체가 같다는 주장을 구분한다.
assert 0.5*np.log(2*np.pi*0.01) < 0	# 좁은 밀도의 봉우리에서는 잔차가 0인 Gaussian NLL이 음수일 수 있다.
gaussian_report = {"n": int(n), "variance": variance, "sse": sse, "mse": mse,
                   "nll": float(gaussian_nll), "independent_reference": independent_gaussian_reference,
                   "identity_error": float(identity_error), "mean_nll": float(gaussian_nll/n)}
print(gaussian_report)

{'n': 3, 'variance': 0.5, 'sse': 1.5, 'mse': 0.5, 'nll': 3.2170948287741004, 'independent_reference': 3.2170948287741004, 'identity_error': 4.440892098500626e-16, 'mean_nll': 1.0723649429247002}


## 5. Bernoulli likelihood에서 BCE 유도

$y_i\in\{0,1\}$, $p_i=\sigma(z_i)$인 독립 표본의 likelihood는
$$\mathcal L=\prod_i p_i^{y_i}(1-p_i)^{1-y_i}.$$
따라서 음의 로그우도는
$$L_{\mathrm{sum}}=-\sum_i\left[y_i\ln p_i+(1-y_i)\ln(1-p_i)\right],\qquad
L_{\mathrm{mean}}=L_{\mathrm{sum}}/n.$$

한 표본에서 $p=\sigma(z)$를 대입하면
$$\ell(z,y)=\log(1+e^z)-yz=\operatorname{logaddexp}(0,z)-yz,\qquad
\frac{\partial\ell}{\partial z}=\sigma(z)-y.$$
sum 손실의 각 logit 미분은 위 식이고 mean 손실의 각 미분은 n으로 나눈다. 앞 역전파 노트북은 단일 표본이어서 평균 계수가 없었다.

`logaddexp`는 exp의 overflow를 피하지만 큰 양수 z와 y=1에서는 마지막 뺄셈이 작은 양의 손실을0으로 상쇄할 수 있다.
아래 binary-label 계산은 수학적으로 같은 $\operatorname{logaddexp}(0,(1-2y)z)$를 사용해 그 작은 값을 보존한다.
미분식도 float64에서 sigmoid가 포화되면 작은 꼬리값을 표현하지 못할 수 있으므로, 충분히 중간인 logit에서 중심차분과 비교한다.

**경계 확률:** $0\log0=0$은 해당 항의 연속 극한이다. p=0,y=0 또는 p=1,y=1의 손실은0이다.
불가능한 사건(p=0,y=1 또는 p=1,y=0)은 손실 $+\infty$다. `0*log(0)`으로 NaN을 만든 뒤 숨기거나 임의 clipping으로 다른 손실을 만들지 않는다.

In [6]:
def _binary_pair(y, values, name):
    """동일하고 비어 있지 않은 shape의 유한 실수 배열과 이진 정답을 반환한다.

    스칼라의 shape는 ()다. 잘못된 자료형은 TypeError를 발생시킨다.
    불일치하거나 빈 shape, 비유한 값 또는 {0,1} 밖의 정답은 ValueError다.
    """
    labels = _as_finite_real_array(y, "y")	# 정답의 실수·유한성 계약을 공통 함수로 검사한다.
    data = _as_finite_real_array(values, name)	# 확률이나 logit도 같은 유한 실수 계약으로 읽는다.
    if labels.shape != data.shape or labels.size == 0:	# 빈 입력과 의도하지 않은 broadcasting을 허용하지 않는다.
        raise ValueError("labels and values must have identical nonempty shapes")
    if not np.all((labels == 0) | (labels == 1)):	# 이번 Bernoulli 구현의 정답은 정확히 0 또는 1이다.
        raise ValueError("binary labels must be 0 or 1")
    return labels, data


def probability_bce(y, p):
    """y/p와 같은 shape의 원소별 이진 NLL을 반환하며 경계의 +inf를 보존한다.

    정답은 이진 값이고 p는 [0,1]에 있어야 한다. 관측된 사건의 확률을 먼저
    선택한 뒤 로그를 취하여 0*log(0)을 피한다. 불가능한 관측은 의도적으로
    +inf를 반환하고 clipping하지 않는다. 잘못된 자료형은 TypeError,
    shape·유한성·정답·확률 범위 위반은 ValueError로 거부한다.
    """
    labels, probabilities = _binary_pair(y, p, "p")	# 정답과 확률의 자료형·shape·이진 조건을 먼저 검사한다.
    if np.any((probabilities < 0) | (probabilities > 1)):	# 확률의 경계를 잘라 보정하지 않고 잘못된 입력을 거부한다.
        raise ValueError("p must be in [0,1]")
    observed_probability = np.where(labels == 1, probabilities, 1-probabilities)	# 관측된 사건만 선택해 0과 log(0)의 불필요한 곱을 피한다.
    with np.errstate(divide="ignore"):
        return -np.log(observed_probability)	# 관측 확률 0의 무한 손실을 수학적 경계값으로 보존한다.


def binary_logit_loss(y, z):
    """동일하고 비어 있지 않은 y/z shape의 원소별 이진 logit NLL을 반환한다.

    이진 정답에서 logaddexp(0,(1-2y)*z)는 logaddexp(0,z)-y*z와 수학적으로
    같으며 큰 정답 logit의 작은 양의 손실을 보존한다. 입력은 유한 실수이고
    정답은 이진 값이어야 한다. 잘못된 입력은 예외로 거부하며 꼬리의 손실이
    float64 표현 범위보다 작으면 이 안정식도 0으로 언더플로될 수 있다.
    """
    labels, logits = _binary_pair(y, z, "z")	# 정답과 logit에도 동일한 비어 있지 않은 shape를 요구한다.
    with np.errstate(under="ignore"):
        return np.logaddexp(0.0, (1-2*labels)*logits)	# 정답 부호에 맞춘 안정식으로 큰 양수끼리의 소거를 피한다.


In [7]:
binary_y = np.array([0.0,1.0])
binary_p = np.array([0.3,0.7])
binary_z = np.log(binary_p/(1-binary_p))	# 확률을 log-odds로 바꾸어 같은 예제를 logit 표현으로 검증한다.
probability_losses = probability_bce(binary_y, binary_p)
logit_losses = binary_logit_loss(binary_y, binary_z)
bce_sum = float(np.sum(probability_losses))	# 독립 표본의 음의 로그우도를 합산한다.
bce_mean = float(np.mean(probability_losses))	# 평균 손실은 같은 표본들의 합 손실과 구분해서 기록한다.
assert abs(bce_sum-0.71334988787746476) <= 1e-15
np.testing.assert_allclose(logit_losses, probability_losses, atol=1e-15, rtol=0)
np.testing.assert_allclose(logit_losses, np.logaddexp(0,binary_z)-binary_y*binary_z, atol=1e-15, rtol=0)
assert abs(bce_mean-bce_sum/2) <= 1e-15
analytic_sum_gradient = sigmoid(binary_z)-binary_y	# 합 BCE의 logit 기울기 p-y를 해석식으로 구한다.
numeric_sum_gradient = numerical_gradient(lambda z: float(np.sum(binary_logit_loss(binary_y,z))), binary_z)	# 손실의 합만 호출하는 중심차분으로 해석 기울기를 검산한다.
numeric_mean_gradient = numerical_gradient(lambda z: float(np.mean(binary_logit_loss(binary_y,z))), binary_z)	# 평균 손실에서는 표본 수로 나눈 기울기가 나오는지 별도 검산한다.
np.testing.assert_allclose(numeric_sum_gradient, [0.3,-0.3], atol=1e-8, rtol=0)
np.testing.assert_allclose(numeric_sum_gradient, analytic_sum_gradient, atol=1e-8, rtol=0)
np.testing.assert_allclose(numeric_mean_gradient, analytic_sum_gradient/2, atol=1e-8, rtol=0)
max_gradient_error = float(np.max(np.abs(numeric_sum_gradient-analytic_sum_gradient)))

boundary_losses = probability_bce([0,1,1,0], [0,1,0,1])	# 확실한 관측과 불가능한 관측의 경계 처리를 함께 검사한다.
np.testing.assert_array_equal(boundary_losses[:2], [0,0])	# 관측 확률이 1이면 손실은 정확히 0이어야 한다.
assert np.all(np.isposinf(boundary_losses[2:])) and not np.any(np.isnan(boundary_losses))	# 관측 확률 0은 NaN이 아닌 양의 무한대 손실이어야 한다.
extreme_losses = binary_logit_loss([0,1,1,0], [-1000,1000,-1000,1000])	# 극단 logit에서 맞은 예측과 틀린 예측의 손실을 비교한다.
np.testing.assert_allclose(extreme_losses, [0,0,1000,1000], atol=0, rtol=0)
assert np.all(np.isfinite(extreme_losses))	# 유한 logit의 큰 오답 손실도 유한하게 표현되는지 확인한다.
positive_tail_at_40 = float(binary_logit_loss(1,40))	# 작지만 표현 가능한 정답 꼬리 손실을 별도로 확인한다.
np.testing.assert_allclose(positive_tail_at_40, 4.248354255291589e-18, atol=0, rtol=1e-14)
assert positive_tail_at_40 > 0	# 표현 가능한 양의 손실을 소거 오차로 0으로 만들지 않는다.
cancellation_example = float(np.logaddexp(0.0,40.0)-40.0)	# 큰 항끼리 빼는 동치식의 소거 오차를 반례로 기록한다.
for invalid_y, invalid_p in (([0.5],[0.3]), ([1],[1.1]), ([0,1],[0.2])):
    try:
        probability_bce(invalid_y, invalid_p)
    except ValueError:
        pass
    else:
        raise AssertionError("invalid binary input was accepted")
bce_report = {"y": binary_y.tolist(), "probabilities": binary_p.tolist(), "sum_loss": bce_sum,
              "mean_loss": bce_mean, "logits": binary_z.tolist(),
              "logit_sum_loss": float(np.sum(logit_losses)), "max_gradient_error": max_gradient_error,
              "analytic_sum_gradient": analytic_sum_gradient.tolist(), "numerical_sum_gradient": numeric_sum_gradient.tolist(),
              "extreme_losses": extreme_losses.tolist(), "positive_tail_at_40": positive_tail_at_40,
              "subtractive_formula_at_40": cancellation_example, "boundary_policy_verified": True}
print(bce_report)

{'y': [0.0, 1.0], 'probabilities': [0.3, 0.7], 'sum_loss': 0.7133498878774649, 'mean_loss': 0.35667494393873245, 'logits': [-0.8472978603872036, 0.8472978603872034], 'logit_sum_loss': 0.7133498878774649, 'max_gradient_error': 8.626654945942391e-12, 'analytic_sum_gradient': [0.3, -0.30000000000000004], 'numerical_sum_gradient': [0.3000000000030756, -0.3000000000086267], 'extreme_losses': [0.0, 0.0, 1000.0, 1000.0], 'positive_tail_at_40': 4.248354255291589e-18, 'subtractive_formula_at_40': 0.0, 'boundary_policy_verified': True}


## 6. 다중 클래스 likelihood에서 Cross-Entropy 유도

각 표본의 one-hot 벡터 $y_{ij}$는 실제 클래스 하나에서만1이며 $\sum_j y_{ij}=1$이다.
$p_{ij}=\operatorname{softmax}(z_i)_j$이면 독립 표본 likelihood와 음의 로그는

$$\mathcal L=\prod_i\prod_j p_{ij}^{y_{ij}},\qquad
L_{\mathrm{sum}}=-\sum_i\sum_j y_{ij}\ln p_{ij}=-\sum_i\ln p_{i,c_i}.$$

이는 categorical Cross-Entropy이고, 표본 평균은 n으로 나눈다. 이진 Bernoulli의 두 결과 항에서 나온 BCE와 유도 출발점을 구분한다.
target가0인 클래스의 $0\log0$ 항은0이고 실제 클래스의 확률이0이면 손실은 $+\infty$다.

logit에서 직접 계산할 때 $s=z-\max(z)$로 이동시켜
$$\ell(z,c)=\ln\sum_j e^{s_j}-s_c$$
를 사용한다. 큰 공통 offset을 더한 logsumexp와 target logit을 마지막에 빼는 취소 오차도 줄인다.
한 표본의 logit 미분은 $p-y_{\mathrm{onehot}}$이며, 평균 손실이면 n으로 나눈다.

In [8]:
def _class_vector(values, target, name):
    """비어 있지 않은 유한 실수 클래스 벡터 (K,)와 [0,K)의 정수 정답을 검사한다.

    잘못된 수치/인덱스 자료형은 TypeError다. 차원 불일치, 빈 벡터,
    비유한 값 또는 범위를 벗어난 인덱스는 ValueError로 거부한다.
    """
    vector = _as_finite_real_array(values, name)	# 클래스별 값은 유한한 실수 배열이어야 한다.
    if vector.ndim != 1 or vector.size == 0:	# 이번 구현은 배치가 아닌 단일 표본의 비어 있지 않은 벡터다.
        raise ValueError("class values must be a nonempty vector")
    if isinstance(target,(bool,np.bool_)) or not isinstance(target,(int,np.integer)):	# bool을 정수 인덱스로 잘못 받아들이지 않는다.
        raise TypeError("target must be an integer class index")
    if not 0 <= target < vector.size:	# 정답 인덱스가 실제 클래스 범위에 속하는지 확인한다.
        raise ValueError("target index is out of range")
    return vector, int(target)	# 검증한 벡터와 표준 정수 인덱스를 다음 계산에 전달한다.


def categorical_probability_ce(probabilities, target):
    """정규화된 확률 벡터 (K,)에서 관측 클래스의 스칼라 NLL을 반환한다.

    확률은 [0,1]이고 합이 절대 허용오차 1e-12 이내에서 1이어야 한다.
    정답 클래스 확률이 0이면 +inf를 반환하며 다른 클래스의 0은 기여하지
    않는다. clipping하지 않으며 잘못된 벡터·인덱스·분포 입력은 거부한다.
    """
    vector, index = _class_vector(probabilities,target,"probabilities")	# 확률 계산 전에 단일 클래스 벡터와 정답 인덱스를 검증한다.
    if np.any((vector < 0) | (vector > 1)) or abs(float(vector.sum())-1) > 1e-12:	# 유효 확률 범위와 합 1 조건을 함께 확인한다.
        raise ValueError("probabilities must be nonnegative and sum to one")
    with np.errstate(divide="ignore"):
        return float(-np.log(vector[index]))	# one-hot 관측의 음의 로그우도는 정답 클래스 한 항만 남는다.


def categorical_logit_ce(logits, target):
    """유한 logit 벡터 (K,)에 공통 이동을 적용해 스칼라 CE를 반환한다.

    log(sum(exp(z-max(z))))-(z_target-max(z))를 계산해 지수 overflow와
    큰 공통 상수의 뺄셈 소거를 피한다. 이동한 차분이나 결과를 표현할 수
    없으면 ValueError를 발생시키며 잘못된 벡터/인덱스 입력도 거부한다.
    """
    vector, index = _class_vector(logits,target,"logits")	# logit 경로도 같은 단일 표본/정답 계약을 사용한다.
    with np.errstate(over="ignore", under="ignore"):
        shifted = vector-np.max(vector)	# 가장 큰 logit을 공통으로 빼 지수 overflow를 막는다.
        value = np.log(np.sum(np.exp(shifted)))-shifted[index]	# 정답 항에서도 같은 이동값을 사용해 큰 공통 상수의 소거를 피한다.
    if not np.all(np.isfinite(shifted)) or not np.isfinite(value):	# 이동 차분이나 최종 손실 자체가 표현 불가능한 경우를 거부한다.
        raise ValueError("shifted logits and CE must be representable")
    return float(value)


class_probabilities = np.array([0.2,0.3,0.5])
target_class = 2	# 0부터 세는 세 번째 클래스를 관측 정답으로 지정한다.
probability_ce = categorical_probability_ce(class_probabilities,target_class)
logit_ce = categorical_logit_ce(np.log([2.0,3.0,5.0]),target_class)
assert abs(probability_ce-0.69314718055994531) <= 1e-15
assert abs(logit_ce-probability_ce) <= 1e-15
extreme_ce = categorical_logit_ce([-1000,0,1000],0)	# 정답이 가장 작은 logit인 큰 오답 예제를 안정적으로 계산한다.
assert extreme_ce == 2000.0	# 단순 Softmax 확률의 underflow와 달리 logit CE는 유한한 차이를 유지한다.
assert abs(categorical_logit_ce([1e20,1e20,1e20],1)-np.log(3)) <= 1e-15	# 큰 공통 상수를 가져도 균등한 세 클래스의 손실은 ln(3)이어야 한다.
assert categorical_probability_ce([0,0,1],2) == 0	# 확률 1인 정답 사건의 손실은 정확히 0이다.
assert np.isposinf(categorical_probability_ce([0,0,1],0))	# 정답 확률 0의 무한 손실을 clipping으로 숨기지 않는다.
for invalid, target in (([0.2,0.3],0), ([-0.2,1.2],0), ([0.2,0.3,0.5],3)):
    try:
        categorical_probability_ce(invalid,target)
    except ValueError:
        pass
    else:
        raise AssertionError("invalid categorical input was accepted")
categorical_report = {"probabilities": class_probabilities.tolist(), "target": target_class,
                      "probability_ce": probability_ce, "logit_ce": logit_ce,
                      "extreme_ce": extreme_ce, "boundary_policy_verified": True}
print(categorical_report)

{'probabilities': [0.2, 0.3, 0.5], 'target': 2, 'probability_ce': 0.6931471805599453, 'logit_ce': 0.6931471805599453, 'extreme_ce': 2000.0, 'boundary_policy_verified': True}


## 7. 확인한 연결과 범위

- Gaussian NLL과 MSE의 최적해 동치는 독립 오차·고정 양의 공통 분산 가정에서 성립한다.
- Bernoulli의 음의 로그가 BCE, categorical의 음의 로그가 one-hot Cross-Entropy다.
- 자연로그·손실합/평균·gradient의 n계수를 구분했다.
- 경계 확률의 불가능 사건은 실제로 무한 손실이며 clipping으로 숨기지 않았다. JSON에는 무한 수를 쓰지 않고 경계 검증 여부를 기록한다.
- 수치 안정화도 float64의 표현 한계를 없애지는 않는다. 극소 밀도/확률은0으로 underflow할 수 있다.
- 미분과 손실 검증은 명시한 예제에 대한 것이며 모든 모델이나 데이터의 통계 가정을 보장하지 않는다.

In [9]:
report = {"seed":42, "dtype":"float64", "numpy_version":np.__version__,	# 모든 앞선 assert가 통과한 실측·입력·환경을 같은 보고서 구조로 모은다.
          "all_assertions_passed":True, "normals":normals, "bernoulli":bernoulli,
          "softmax":softmax_report, "gaussian":gaussian_report,
          "bce":bce_report, "categorical":categorical_report,
          "figures":["probability_normal.png","probability_bernoulli.png"],
          "notes":["Normal second parameter is variance, not standard deviation.",
                   "Natural logarithms; sum and mean losses are distinguished.",
                   "Gaussian NLL and MSE share minimizers only under the stated fixed-variance assumptions.",
                   "Probability boundaries are not clipped; impossible-event losses are positive infinity.",
                   "Only finite measurements are serialized; boundary assertions are recorded as booleans."]}
(output_dir / "probability_metrics.json").write_text(	# 기존 결과 파일명과 인코딩을 유지해 문서 링크가 가리키는 산출물을 저장한다.
    json.dumps(report,ensure_ascii=False,indent=2,allow_nan=False)+"\n",encoding="utf-8",newline="\n")
print("PASS: distributions, normalization, MLE identities, boundary policies, and logit gradients")
print("Saved probability_normal.png, probability_bernoulli.png, probability_metrics.json")

PASS: distributions, normalization, MLE identities, boundary policies, and logit gradients
Saved probability_normal.png, probability_bernoulli.png, probability_metrics.json
